In [10]:
from google.colab import drive
import os

# Grab the paths to the metadata files and the waves file from google drive
drive.mount("/content/drive")
BASE_DIR = "/content/drive/My Drive/Computer Science/Jabbel/data"
csv_path = os.path.join(BASE_DIR, "metadata.csv")
audio_dir = os.path.join(BASE_DIR, "wavs")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Dataset
```python
datasets.load_dataset()
```
This function is from HuggingFace that loads in audio dir based on the file names e.g (metedata.csv and wavs dir)

In [11]:
import datasets

dataset = datasets.load_dataset(
    "csv",
    data_files = csv_path,
    # Tells HF the divider that seperates the ID : Transcript... (normally a , but it is a | for me) 
    sep = "|",                                      
     # Tells PyTorch the headers ID : Transcript : Norm_Trans for each record in the csv, as there's no headers
    column_names=["file_id", "transcription", "normalized_transcription"],   
    split = "train"                 
)

Generating train split: 0 examples [00:00, ? examples/s]

# Create Full Paths to .wav files
Because the csv file only lists the ID and not the file extention, I need to map over the csv and manually add them so the ful path is built
1. <code>batch</code>: This takes mutiple rows at a time, instead of looking at one row at a time.
2. <code>f"{fid}.wav"</code>: This takes the ID and ads .wav onto the end of each ID
3. <code>batch["audio"]</code>: This adds a new column called audio with all the correct path names
4. <code>dataset.map</code>: This tells the dataset to run the function across every single row in the dataset


In [12]:
def add_audio_paths(batch):
    batch["audio"] = [os.path.join(audio_dir, f"{fid}.wav") for fid in batch["file_id"]]
    return batch

dataset = dataset.map(add_audio_paths, batched = True)

Map:   0%|          | 0/13100 [00:00<?, ? examples/s]

# Cast The Path Strings To Actual Audio Files
1. Makes the audio column, currently just text filenames, be treatd as actual audio rather than just plane text
2. <code>Audio(sampling_rate = 16_000)</code>: Tells the model no matter what make alll these files 16,000 Hz to make sure they are all the same Hz

In [18]:
dataset = dataset.cast_column("audio", datasets.Audio(sampling_rate = 16_000))

print(dataset[0]["audio"])
print(dataset[0])

{'file_id': 'LJ001-0001', 'transcription': 'Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition', 'normalized_transcription': 'Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7bf3691d7c20>}
